In [1]:
import torch
import torch.nn as nn

In [2]:
class SeqToVecRNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, num_layers):
        super().__init__()
        self.rnn = nn.RNN(input_size, hidden_size, num_layers=num_layers, batch_first=True)
        self.output = nn.Linear(hidden_size, output_size)

    def forward(self, X):
        outputs, last_state = self.rnn(X)
        last_step = outputs[:, -1]
        return self.output(last_step)

In [3]:
class SeqToSeqRNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, num_layers):
        super().__init__()
        self.rnn = nn.RNN(input_size, hidden_size, num_layers=num_layers, batch_first=True)
        self.output = nn.Linear(hidden_size, output_size)

    def forward(self, X):
        outputs, last_state = self.rnn(X)
        return self.output(outputs)

In [4]:
class VecToSeqRNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, num_layers, seq_len):
        super().__init__()
        self.seq_len = seq_len
        self.rnn = nn.RNN(input_size, hidden_size, num_layers=num_layers, batch_first=True)
        self.output = nn.Linear(hidden_size, output_size)

    def forward(self, x_vec):
        X_repeated = x_vec.unsqueeze(1).repeat(1, self.seq_len, 1)
        outputs, last_state = self.rnn(X_repeated)
        return self.output(outputs)

In [ ]:
class EncoderDecoderRNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, num_layers, seq_len):
        super().__init__()
        self.hidden_size = hidden_size
        self.seq_len = seq_len
        self.encoder = nn.RNN(input_size, hidden_size, num_layers=num_layers, batch_first=True)
        self.decoder = nn.RNN(hidden_size, hidden_size, num_layers=num_layers, batch_first=True)
        self.output = nn.Linear(hidden_size, output_size)

    def forward(self, X):
        _, last_state = self.encoder(X)
        batch_size = X.shape[0]
        decoder_input = torch.zeros(batch_size, self.seq_len, self.hidden_size)
        outputs, _ = self.decoder(decoder_input, last_state)
        return self.output(outputs)